# 03b — US Pipeline (2014 Sample)

**Thesis:** Implied Volatility Smile Spillovers (AP-33)  
**Author:** Başar Hacımustafaoğlu — 1******6

Same idea as 03a, but for the US 2014 window. Apple options, IvyDB US via WRDS sample.

Input: `data/raw/window_us_2014/`  
Output: `data/intermediate/us2014_analysis_ready__<timestamp>.csv`

Make sure notebook 02 has been run before this — we need the raw CSVs to exist.

## One important difference vs 03a — the US delta convention

IvyDB US stores deltas differently from IvyDB Europe. In the US table:
- Both calls and puts have **positive deltas** (20 to 80)
- You tell them apart using the `cp_flag` column (`'C'` or `'P'`), not the sign of delta

So when we want the 25-delta put, we look for `cp_flag == 'P'` and `delta == 25` — not `-25`.

This is the opposite of the EU convention, where puts had negative deltas. Keep this in mind through Step 4.

Smile parameters (same definitions as 03a, Malz 1997)):

| Parameter | US lookup |
|---|---|
| ATM IV | call, delta = 50 |
| Skew | put(delta=25) − call(delta=75) |
| Curvature | put(delta=25) + call(delta=75) − 2 × ATM |

The formulas are identical to 03a. Only how we find the puts changes.

## Step 1 — Imports, paths, constants

In [1]:
import os
import glob
import datetime
import json
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

RAW_US       = "../data/raw/window_us_2014"
INTERMEDIATE = "../data/intermediate"
LOG_DIR      = "../logs"

os.makedirs(INTERMEDIATE, exist_ok=True)
os.makedirs(LOG_DIR,      exist_ok=True)

TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# OptionMetrics sentinel — same as EU
IV_SENTINEL = -99

# Apple in the US sample
# We'll confirm this after loading security_name
US_SECURITYID = None  # set after Step 2

transform_log = []

def log_transform(step, table, before, after, reason):
    entry = {
        "step":    step,
        "table":   table,
        "before":  before,
        "after":   after,
        "dropped": before - after,
        "reason":  reason,
    }
    transform_log.append(entry)
    print(f"  [{step}] {table}: {before} → {after} rows (dropped {before - after}: {reason})")

print(f"Timestamp : {TIMESTAMP}")
print(f"RAW_US    : {RAW_US}")
print(f"INTERMEDIATE: {INTERMEDIATE}")
print("\nReady.")

Timestamp : 20260314_142601
RAW_US    : ../data/raw/window_us_2014
INTERMEDIATE: ../data/intermediate

Ready.


## Step 2 — Load raw US 2014 CSVs

Same loader as 03a. Expected row counts from notebook 02:
- `volatility_surface_2014`: 2,600 rows
- `option_price_2014`: varies
- `security_price`: ~50 rows
- `historical_volatility`: ~130 rows
- `security_name`: 1 row
- `frb.rates_daily`: ~120 rows
- `crsp.dsi`: 82 rows
- `ff.factors_daily`: 82 rows
- `cboe.cboe`: 82 rows
- `comp.g_exrt_dly`: ~20,000 rows

If any count looks very different from the above, stop and check before continuing.

In [3]:
print("Loading US 2014 raw files...\n")

df_surface  = load_latest("optionmsamp_us__vsurfd2014")
df_optprice = load_latest("optionmsamp_us__opprcd2014")
df_secprice = load_latest("optionmsamp_us__secprd")
df_secname  = load_latest("optionmsamp_us__secnmd")

df_frb      = load_latest("frb__rates_daily")
df_dsi      = load_latest("crsp__dsi")
df_ff       = load_latest("ff__factors_daily")
df_vix      = load_latest("cboe__cboe")
df_fx       = load_latest("comp__g_exrt_dly")

print("\nAll files loaded.")

Loading US 2014 raw files...

✓ optionmsamp_us__vsurfd2014__US2014__20260314_142434.csv: 2600 rows x 9 cols
✓ optionmsamp_us__opprcd2014__US2014__20260314_142434.csv: 36786 rows x 22 cols
✓ optionmsamp_us__secprd__US2014__20260314_142434.csv: 10 rows x 11 cols
✓ optionmsamp_us__secnmd__US2014__20260314_142434.csv: 3 rows x 8 cols
✓ frb__rates_daily__US2014__20260314_142434.csv: 120 rows x 83 cols
✓ crsp__dsi__US2014__20260314_142434.csv: 82 rows x 11 cols
✓ ff__factors_daily__US2014__20260314_142434.csv: 82 rows x 6 cols
✓ cboe__cboe__US2014__20260314_142434.csv: 82 rows x 17 cols
✓ comp__g_exrt_dly__US2014__20260314_142434.csv: 20728 rows x 5 cols

All files loaded.


In [5]:
# Confirm which security we're working with
print("Security in this sample:")
print(df_secname.to_string())

# Set the security ID from what we find above
# Should be Apple — securityid will be set after we confirm
US_SECURITYID = df_secname['secid'].values[0]
print(f"\nUS_SECURITYID set to: {US_SECURITYID}")

Security in this sample:
          secid effect_date    cusip ticker  class              issuer issue         sic
0 101594.000000  1996-01-02  3783310   AAPL    NaN  APPLE COMPUTER INC   COM         NaN
1 101594.000000  2000-11-28  3783310   AAPL    NaN  APPLE COMPUTER INC   COM 3571.000000
2 101594.000000  2007-01-11  3783310   AAPL    NaN           APPLE INC   COM 3571.000000

US_SECURITYID set to: 101594.0


## Step 3 — Clean the volatility surface

Same cleaning logic as 03a:
- Parse dates
- Filter to our security
- Drop sentinel IVs (-99)
- Drop non-positive IVs
- Inspect the delta and days grid

The grid inspection at the end is important — we need to confirm the US delta convention before Step 4.

In [9]:
print("STEP 3 — Clean volatility surface\n")

surf = df_surface.copy()
print(f"Starting rows: {len(surf)}")

# 3a. Parse dates
surf['date'] = pd.to_datetime(surf['date'])
print(f"\n[3a] Date range: {surf['date'].min().date()} → {surf['date'].max().date()}")
print(f"     Unique trading days: {surf['date'].nunique()}")

# 3b. Filter to our security
n_before = len(surf)
surf = surf[surf['secid'] == US_SECURITYID]
log_transform("3b", "surface", n_before, len(surf),
              f"keep secid={US_SECURITYID} only")

# 3c. Drop sentinel IVs
n_before = len(surf)
surf = surf[surf['impl_volatility'] != IV_SENTINEL]
log_transform("3c", "surface", n_before, len(surf),
              "drop IV sentinel -99 (inversion failure)")

# 3d. Drop non-positive IVs
n_before = len(surf)
surf = surf[surf['impl_volatility'] > 0]
log_transform("3d", "surface", n_before, len(surf),
              "drop IV <= 0 (economically invalid)")

# 3e. Inspect the grid
print(f"\n[3e] Delta nodes present : {sorted(surf['delta'].unique())}")
print(f"     Days nodes present   : {sorted(surf['days'].unique())}")
print(f"     cp_flag values       : {surf['cp_flag'].unique()}")
print(f"     Unique trading days  : {surf['date'].nunique()}")
print(f"\nSurface after cleaning: {len(surf)} rows")

print("""
IMPORTANT — delta convention confirmed from data:
Puts have NEGATIVE deltas (-80 to -20), calls have POSITIVE deltas (+20 to +80).
This matches the EU convention, NOT what the notebook markdown said.
Step 4 must use delta == -25 for puts, delta == 75 for calls.
""")

STEP 3 — Clean volatility surface

Starting rows: 2600

[3a] Date range: 2014-03-03 → 2014-03-14
     Unique trading days: 10
  [3b] surface: 2600 → 2600 rows (dropped 0: keep secid=101594.0 only)
  [3c] surface: 2600 → 2600 rows (dropped 0: drop IV sentinel -99 (inversion failure))
  [3d] surface: 2600 → 2600 rows (dropped 0: drop IV <= 0 (economically invalid))

[3e] Delta nodes present : [np.float64(-80.0), np.float64(-75.0), np.float64(-70.0), np.float64(-65.0), np.float64(-60.0), np.float64(-55.0), np.float64(-50.0), np.float64(-45.0), np.float64(-40.0), np.float64(-35.0), np.float64(-30.0), np.float64(-25.0), np.float64(-20.0), np.float64(20.0), np.float64(25.0), np.float64(30.0), np.float64(35.0), np.float64(40.0), np.float64(45.0), np.float64(50.0), np.float64(55.0), np.float64(60.0), np.float64(65.0), np.float64(70.0), np.float64(75.0), np.float64(80.0)]
     Days nodes present   : [np.float64(30.0), np.float64(60.0), np.float64(91.0), np.float64(122.0), np.float64(152.0), np.

In [8]:
# Fixing the variable names for the cell above
print(df_surface.columns.tolist())
print(df_surface.head(3).to_string())

['secid', 'date', 'days', 'delta', 'impl_volatility', 'impl_strike', 'impl_premium', 'dispersion', 'cp_flag']
          secid        date      days      delta  impl_volatility  impl_strike  impl_premium  dispersion cp_flag
0 101594.000000  2014-03-03 30.000000 -80.000000         0.230443   559.197100     35.150580    0.028626       P
1 101594.000000  2014-03-03 30.000000 -75.000000         0.221934   552.070300     29.101250    0.014204       P
2 101594.000000  2014-03-03 30.000000 -70.000000         0.217388   546.410500     24.649490    0.007244       P


## Step 4 — Compute smile parameters per day

Same three parameters as 03a (Malz 1997):
- ATM IV = call at delta 50
- Skew = put(25) − call(75)
- Curvature = put(25) + call(75) − 2 × ATM

The only difference from 03a: we look for `delta == 25` in puts (positive), not `delta == -25`.
The cp_flag column separates calls from puts.

In [10]:
# First check the exact cp_flag values before splitting
print("cp_flag value counts:")
print(surf['cp_flag'].value_counts())
print(f"\nUnique cp_flag values: {surf['cp_flag'].unique()}")
print("\nExpected: 'C' and 'P' — update CALL_FLAG/PUT_FLAG below if different.")

cp_flag value counts:
cp_flag
P    1300
C    1300
Name: count, dtype: int64

Unique cp_flag values: ['P' 'C']

Expected: 'C' and 'P' — update CALL_FLAG/PUT_FLAG below if different.


In [12]:
print("STEP 4 — Compute smile parameters\n")

CALL_FLAG = 'C'
PUT_FLAG  = 'P'

calls = surf[surf['cp_flag'] == CALL_FLAG].copy()
puts  = surf[surf['cp_flag'] == PUT_FLAG].copy()

print(f"Calls: {len(calls)} rows")
print(f"Puts : {len(puts)} rows")

print(f"\nCall delta range: {sorted(calls['delta'].unique())}")
print(f"Put  delta range: {sorted(puts['delta'].unique())}")

smile_records = []

dates     = sorted(surf['date'].unique())
days_list = sorted(surf['days'].unique())

for date in dates:
    for days in days_list:

        c = calls[(calls['date'] == date) & (calls['days'] == days)]
        p = puts[ (puts['date']  == date) & (puts['days']  == days)]

        def get_iv(df, delta):
            """Get IV at a specific delta node. int() cast avoids np.int64 type mismatches."""
            row = df[df['delta'] == int(delta)]
            if len(row) == 1:
                return float(row['impl_volatility'].values[0])
            return np.nan

        # vsurfd2014 delta convention: calls positive (+50, +75), puts negative (-25)
        atm_iv = get_iv(c,  50)
        call75 = get_iv(c,  75)
        put25  = get_iv(p, -25)

        skew      = put25 - call75 \
                    if not (np.isnan(put25) or np.isnan(call75)) else np.nan
        curvature = (put25 + call75 - 2 * atm_iv) \
                    if not (np.isnan(put25) or np.isnan(call75) or np.isnan(atm_iv)) else np.nan

        smile_records.append({
            'date':      date,
            'days':      days,
            'atm_iv':    atm_iv,
            'skew':      skew,
            'curvature': curvature,
            'put25_iv':  put25,
            'call75_iv': call75,
        })

df_smile = pd.DataFrame(smile_records)

print(f"\nSmile parameters computed: {len(df_smile)} (date, days) cells")
print(f"Missing ATM IV   : {df_smile['atm_iv'].isna().sum()}")
print(f"Missing skew     : {df_smile['skew'].isna().sum()}")
print(f"Missing curvature: {df_smile['curvature'].isna().sum()}")
print(f"\nSample (first 10 rows):")
print(df_smile.head(10).to_string(index=False))

print("""
Sanity checks:
- ATM IV should be in a plausible range for Apple (maybe 20-40% annualised)
- Skew for US equity options is typically negative (left tail more expensive)
- If skew or curvature are all NaN, the delta lookup failed — check delta signs above
""")

STEP 4 — Compute smile parameters

Calls: 1300 rows
Puts : 1300 rows

Call delta range: [np.float64(20.0), np.float64(25.0), np.float64(30.0), np.float64(35.0), np.float64(40.0), np.float64(45.0), np.float64(50.0), np.float64(55.0), np.float64(60.0), np.float64(65.0), np.float64(70.0), np.float64(75.0), np.float64(80.0)]
Put  delta range: [np.float64(-80.0), np.float64(-75.0), np.float64(-70.0), np.float64(-65.0), np.float64(-60.0), np.float64(-55.0), np.float64(-50.0), np.float64(-45.0), np.float64(-40.0), np.float64(-35.0), np.float64(-30.0), np.float64(-25.0), np.float64(-20.0)]

Smile parameters computed: 100 (date, days) cells
Missing ATM IV   : 0
Missing skew     : 0
Missing curvature: 0

Sample (first 10 rows):
      date       days   atm_iv     skew  curvature  put25_iv  call75_iv
2014-03-03  30.000000 0.204373 0.013537   0.007033  0.214658   0.201121
2014-03-03  60.000000 0.225857 0.010593   0.010953  0.236630   0.226037
2014-03-03  91.000000 0.234899 0.009087   0.010617  0.24

## Step 5 — Clean and align control variables

Same structure as 03a. We build the trading-day calendar from crsp.dsi and align everything to it.

One note on EUR/USD: the FX table only has GBP as the base currency, so we construct EUR/USD as (GBP/USD) / (GBP/EUR) — same triangular arithmetic as 03a.

In [14]:
print("STEP 5 — Build trading-day calendar and clean controls\n")

# 5a. Trading-day calendar from crsp.dsi
cal = df_dsi.copy()
cal['date'] = pd.to_datetime(cal['date'])
trading_days = sorted(cal['date'].unique())
print(f"[5a] Trading days: {len(trading_days)}")
print(f"     From: {min(trading_days).date()} → {max(trading_days).date()}")

# 5b. FRB rates
frb = df_frb.copy()
frb['date'] = pd.to_datetime(frb['date'])
frb = frb[frb['date'].isin(trading_days)]
frb = frb[['date', 'dff', 'dgs1', 'dgs10', 't10y2y', 'bamlh0a0hym2']].copy()
frb = frb.rename(columns={
    'dff':          'us_rf_rate',
    'dgs1':         'us_yield_1y',
    'dgs10':        'us_yield_10y',
    't10y2y':       'us_term_spread',
    'bamlh0a0hym2': 'us_hy_spread',
})
frb = frb.sort_values('date').reset_index(drop=True)
print(f"\n[5b] FRB rates: {len(frb)} rows, {frb.isna().sum().sum()} total NaNs")

# 5c. CRSP market index
dsi = df_dsi.copy()
dsi['date'] = pd.to_datetime(dsi['date'])
dsi = dsi[['date', 'sprtrn', 'spindx']].copy()
dsi = dsi.rename(columns={'sprtrn': 'sp500_ret', 'spindx': 'sp500_idx'})
dsi = dsi.sort_values('date').reset_index(drop=True)
print(f"[5c] CRSP DSI: {len(dsi)} rows, {dsi.isna().sum().sum()} total NaNs")

# 5d. Fama-French factors
ff = df_ff.copy()
ff['date'] = pd.to_datetime(ff['date'])
ff = ff[ff['date'].isin(trading_days)]
ff = ff[['date', 'mktrf', 'smb', 'hml', 'rf']].copy()
ff = ff.rename(columns={
    'mktrf': 'ff_mktrf',
    'smb':   'ff_smb',
    'hml':   'ff_hml',
    'rf':    'ff_rf',
})
ff = ff.sort_values('date').reset_index(drop=True)
print(f"[5d] FF factors: {len(ff)} rows, {ff.isna().sum().sum()} total NaNs")

# 5e. VIX
vix = df_vix.copy()
vix['date'] = pd.to_datetime(vix['date'])
vix = vix[vix['date'].isin(trading_days)]
vix = vix[['date', 'vix']].copy()
vix = vix.sort_values('date').reset_index(drop=True)
print(f"[5e] VIX: {len(vix)} rows, {vix.isna().sum().sum()} total NaNs")

# 5f. EUR/USD via GBP cross rate
# comp.g_exrt_dly only has GBP as base, so EUR/USD = GBP/USD / GBP/EUR
fx_raw = df_fx.copy()
fx_raw['date'] = pd.to_datetime(fx_raw['datadate'])
fx_raw = fx_raw[fx_raw['exrattpd'] == 'AR']  # actual rate only, not conversion factor

gbp_usd = fx_raw[fx_raw['tocurd'] == 'USD'][['date', 'exratd']].rename(columns={'exratd': 'gbp_usd'})
gbp_eur = fx_raw[fx_raw['tocurd'] == 'EUR'][['date', 'exratd']].rename(columns={'exratd': 'gbp_eur'})

fx_cross = gbp_usd.merge(gbp_eur, on='date', how='inner')
fx_cross['eurusd'] = fx_cross['gbp_usd'] / fx_cross['gbp_eur']
fx_cross = fx_cross[fx_cross['date'].isin(trading_days)]
fx = fx_cross[['date', 'eurusd']].sort_values('date').reset_index(drop=True)
print(f"[5f] EUR/USD: {len(fx)} rows, {fx.isna().sum().sum()} total NaNs")
print(f"     Mean: {fx['eurusd'].mean():.4f} | Range: {fx['eurusd'].min():.4f} → {fx['eurusd'].max():.4f}")
print("     EUR/USD for early 2014 should be around 1.35-1.39 — sanity check this")

# 5g. Historical vol — not pulled in notebook 02 for US sample, skipped.
# Unrelevant for the current regression we will run after
print("[5g] Historical vol: not available in US 2014 sample — skipped")
hv30 = pd.DataFrame(columns=['date', 'hist_vol_30d'])

print("\nStep 5 complete.")

STEP 5 — Build trading-day calendar and clean controls

[5a] Trading days: 82
     From: 2014-01-02 → 2014-04-30

[5b] FRB rates: 82 rows, 0 total NaNs
[5c] CRSP DSI: 82 rows, 0 total NaNs
[5d] FF factors: 82 rows, 0 total NaNs
[5e] VIX: 82 rows, 0 total NaNs
[5f] EUR/USD: 82 rows, 0 total NaNs
     Mean: 1.3731 | Range: 1.3484 → 1.3927
     EUR/USD for early 2014 should be around 1.35-1.39 — sanity check this
[5g] Historical vol: not available in US 2014 sample — skipped

Step 5 complete.


## Step 6 — Merge into one daily panel

Left join everything onto the smile table. Row count should stay at whatever df_smile has — any change means something went wrong in a merge.

In [ ]:
print("STEP 6 — Merge into daily panel\n")

panel = df_smile.copy()
print(f"Base (smile): {len(panel)} rows")

merges = [
    (frb, 'date', 'FRB rates'),
    (dsi, 'date', 'CRSP S&P500'),
    (ff,  'date', 'FF factors'),
    (vix, 'date', 'VIX'),
    (fx,  'date', 'EUR/USD'),
]

for df_ctrl, key, label in merges:
    n_before = len(panel)
    panel = panel.merge(df_ctrl, on=key, how='left')
    n_after = len(panel)
    if n_before != n_after:
        print(f"  WARNING: row count changed after merging {label}: {n_before} → {n_after}")
    else:
        print(f"  Merged {label}: {n_after} rows")

panel = panel.sort_values(['date', 'days']).reset_index(drop=True)

print(f"\nFinal panel shape: {panel.shape}")
print(f"Columns: {list(panel.columns)}")

## Step 7 — Validation

Check everything looks right before saving.

In [ ]:
print("STEP 7 — Final validation\n")

print(f"[1] Shape: {panel.shape[0]:,} rows x {panel.shape[1]} columns")

print(f"\n[2] Date coverage:")
print(f"    {panel['date'].min().date()} → {panel['date'].max().date()}")
print(f"    Unique trading days: {panel['date'].nunique()}")
print(f"    Unique maturity nodes (days): {sorted(panel['days'].unique())}")

print(f"\n[3] Missingness by column:")
miss = panel.isna().sum()
miss = miss[miss > 0]
if len(miss) == 0:
    print("    No missing values.")
else:
    for col, n in miss.items():
        print(f"    {col:<25} {n:>5} missing ({100*n/len(panel):.1f}%)")

print(f"\n[4] Duplicate rows: {panel.duplicated().sum()}")

print(f"\n[5] Key variable summary stats:")
key_cols = ['atm_iv', 'skew', 'curvature', 'us_rf_rate', 'vix', 'eurusd', 'sp500_ret']
available = [c for c in key_cols if c in panel.columns]
print(panel[available].describe().round(4).to_string())

print(f"\n[6] Transformation log:")
for entry in transform_log:
    print(f"    [{entry['step']}] {entry['table']}: dropped {entry['dropped']} rows — {entry['reason']}")

print("""
Expected sanity checks for US 2014 window:
- ATM IV for Apple: probably 20-40% range
- Skew: negative (equity left tail)
- VIX: 2014 was relatively calm, expect ~12-17 range
- EUR/USD: early 2014 was around 1.35-1.39
- SP500 returns: small daily numbers around 0
""")

## Step 8 — Save

In [ ]:
print("STEP 8 — Save\n")

out_path = os.path.join(INTERMEDIATE, f"us2014_analysis_ready__{TIMESTAMP}.csv")
panel.to_csv(out_path, index=False)
print(f"Panel saved: {out_path}")
print(f"Shape: {panel.shape[0]:,} rows x {panel.shape[1]} columns")

log_path = os.path.join(LOG_DIR, f"us2014_pipeline_log__{TIMESTAMP}.json")
with open(log_path, 'w') as f:
    json.dump(transform_log, f, indent=2, default=str)
print(f"Transform log saved: {log_path}")

print("\nNotebook 03b complete.")
print("\nNext: notebook 04 — combine EU and US panels, build regression dataset.")